# Практикум 06 · Калибровка и дополненная реальность

Ниже находится полная глава курса, затем — ячейки практики. Прочитайте главу перед изменением параметров.

Стандартный **Run All** использует учебные изображения или синтетические данные. Подготовьте Python с OpenCV и NumPy по стартовой инструкции.


# 9.6. Дополненная реальность: шахматная доска и положение камеры

**Работа на нашем роботе:** создайте [личную копию курса и выберите нужное окружение](../docs/SETUP_ROBOT.md). Все команды ниже выполняйте из её корня; результаты `outputs/` будут принадлежать вашей копии. Самостоятельный поиск камеры, контейнера и ROS-топиков разобран в той же инструкции.

[Назад: 9.5](../lessons/05_applications.md) · [Оглавление](../README.md) · [Далее: 9.7](../lessons/07_markers.md)

**Результат:** изображение с плоской виртуальной накладкой; после собственной калибровки — оси координат доски и оценка её положения. Для базовой части достаточно ноутбука. Для метрической части нужны камера, плоская шахматная доска и линейка. Ориентир: одно занятие на геометрию, отдельное — на калибровку.

## 1. Сначала получите понятный результат

Работайте из корня репозитория в окружении OpenCV:

```bash
python3 -m rosmaster_cv.lesson06_ar --synthetic
```

Откройте `outputs/06/checkerboard.png` и `outputs/06/ar_overlay.png`. Программа находит **7 × 5 внутренних углов**, уточняет их координаты и накладывает прямоугольник на плоскость доски. В `result.json` находятся матрица преобразования и признак `metric_pose_available: false`.

Внутренний угол расположен на пересечении четырёх клеток. У доски с 8 × 6 клетками получается 7 × 5 внутренних углов. Ошибка в этом подсчёте — частая причина сообщения `Chessboard not found`.

Преобразование плоской накладки задаётся гомографией: для координат изображения выполняется соответствие `p' ~ H p`. Оно позволяет расположить рисунок на видимой плоскости. Для определения расстояния до настоящей камеры необходимы дополнительные данные: её калибровка и измеренный размер клеток.

## 2. Подготовьте настоящую доску

Создайте ровный шаблон:

```bash
python3 -m rosmaster_cv.lesson06_ar --board-template --columns 9 --rows 6
```

Распечатайте `outputs/06/checkerboard_template.png`, закрепите на жёсткой плоской основе. После печати измерьте сторону клетки линейкой. Масштаб принтера может изменить размер: число пикселей шаблона не задаёт миллиметры на бумаге. Сохраните измерение в отчёте.

Получите 12–20 фотографий этой доски **той камерой и с тем разрешением, которые будут использоваться в AR**. Меняйте наклон, расстояние и положение в кадре, включая края изображения. Доска должна полностью помещаться в кадр и оставаться резкой. Десять одинаковых кадров почти не добавляют информации.

Сохраните снимки в `data/calibration/`. При рабочем Python 3 `cv_bridge` способ получения кадра показан в [уроке 9.8](../lessons/08_ros_image.md); можно заранее пройти его первый запуск. На ноутбуке подойдут снимки с подключённой камеры. Имена файлов сделайте разными.

## 3. Вычислите параметры камеры

Следующая команда предполагает, что **вы измерили клетку ровно 0.02 м**. Подставьте своё значение:

```bash
python3 -m rosmaster_cv.lesson06_ar --calibrate-dir data/calibration --columns 9 --rows 6 --square-size 0.02
```

Требуются минимум десять кадров с найденными углами. Результат `outputs/06/calibration.json` содержит `camera_matrix` — матрицу K, коэффициенты дисторсии D, разрешение и ошибки репроекции. Список `rejected_images` показывает снимки, на которых доска не найдена.

Небольшая ошибка RMS означает согласование наблюдаемых и спроецированных углов. Дополнительно проверьте результат на новом кадре и известном расстоянии: одного RMS недостаточно, чтобы подтвердить метрическую точность.

## 4. Добавьте оси на новый снимок

Положите проверочный кадр в `data/board_test.png`, затем выполните:

```bash
python3 -m rosmaster_cv.lesson06_ar --input data/board_test.png --columns 9 --rows 6 --square-size 0.02 --calibration outputs/06/calibration.json --output-dir outputs/06/metric
```

`solvePnP` оценивает поворот и перенос доски относительно камеры. `tvec_m` выражен в метрах благодаря измеренной стороне клетки. Начало координат — первый найденный внутренний угол; на симметричной доске его выбор может изменяться при развороте. Изменение разрешения, фокуса или камеры требует проверки калибровки. Скрипт отклоняет разрешение, отличающееся от записанного.

## 5. Заводская реализация ROSMASTER

В проверенной главе Yahboom используется ROS1-пакет `yahboomcar_visual`, параметры камеры лежат в `AR/astra.yaml`. Для заводской калибровки приведён пример:

```bash
roslaunch usb_cam usb_cam-test.launch
```

В другом терминале:

```bash
rosrun camera_calibration cameracalibrator.py image:=/usb_cam/image_raw camera:=/usb_cam --size 9x6 --square 0.02
```

Топик, количество внутренних углов и размер здесь также должны соответствовать вашим данным. После подготовки заводской калибровки запустите:

```bash
roslaunch yahboomcar_visual simple_AR.launch display:=true flip:=false
```

Клавиша `f` переключает фигуры, `q` завершает окно. Не запускайте второй драйвер для уже занятой камеры; порядок включения проверьте в своём launch-файле. JSON учебного скрипта и заводской YAML — разные форматы, их нельзя просто переименовать.

## Задание и проверка

Сдайте исходный кадр, накладку, `result.json` и краткие ответы: почему нужны внутренние углы; что определяет масштаб `tvec`; почему нельзя калибровать настоящую камеру синтетической картинкой. Для метрической части приложите размер клетки, число кадров, RMS и проверку на новом снимке. Успех базовой части — 35 найденных углов и накладка внутри доски; успех метрической — воспроизводимый результат на новых кадрах с объяснёнными единицами.

**Источники:** [Yahboom 9.6](https://www.yahboom.net/study/ROSMASTER-X3-PLUS/8147/), [OpenCV: калибровка и положение доски](https://docs.opencv.org/4.10.0/dc/d43/tutorial_camera_calibration_square_chess.html), [OpenCV: оценивание позы](https://docs.opencv.org/4.10.0/d7/d53/tutorial_py_pose.html).


---

## Ячейки практики

Запускайте последовательно. Команды используют тот же Python, что и ядро ноутбука. Рабочая папка дочерних процессов — корень репозитория. Ошибка команды останавливает ячейку и показывает причину.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Jupyter may start in the repository root or in its notebooks directory.
ROOT = None
for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (candidate / "rosmaster_cv").is_dir() and (candidate / "lessons").is_dir():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из корня репозитория или его папки notebooks.")

def run_module(module, *arguments):
    command = [sys.executable, "-m", "rosmaster_cv." + module]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

def run_tool(filename, *arguments):
    command = [sys.executable, str(ROOT / "tools" / filename)]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

print("Корень курса:", ROOT)
print("Python ядра:", sys.executable)


In [ ]:
run_module("lesson06_ar", "--synthetic")


In [ ]:
def show_results(relative_directory, limit=4):
    directory = ROOT / relative_directory
    paths = sorted(directory.glob("*.png"))[:limit]
    if not paths:
        print("Изображений пока нет:", directory)
        return
    try:
        from IPython.display import Image, display
    except ImportError:
        for path in paths:
            print(path)
        return
    for path in paths:
        print(path.relative_to(ROOT))
        display(Image(filename=str(path)))

show_results("outputs/06")


Синтетический пример проверяет последовательность операций. Для измерений на камере выполните реальную калибровку из главы; размеры и матрицу камеры берите из своих данных.


## Результат самостоятельной работы

Выполните задания и критерии приёмки из главы. Сохраните созданные изображения и JSON, параметры запуска и краткие ответы в своём отчёте. Повторите опыт с собственными данными после успешного учебного запуска.
